In [9]:
import numpy as np
from astropy.io import fits
from astropy.wcs import WCS
import astropy.units as u
from reproject import reproject_interp

# =========================
# 1. 读入原始 Galactic 图像
# =========================
infile = "hi4pi-hvc-nhi-gal-car.fits"
hdul = fits.open(infile)
data = hdul[0].data
hdr  = hdul[0].header
wcs_gal = WCS(hdr)

# 将NHI单位从log(10^18 cm^-2)转换为K
data = 10**data / 1.823e18

In [10]:

# =========================
# 2. 赤道坐标范围（FK5）
# =========================
ra_min, ra_max   = 60.0, 80.0   # deg
dec_min, dec_max = -13.0, 2.0   # deg

# =========================
# 3. 像素尺度（沿用 HI4PI）
# =========================
pixscale = abs(hdr["CDELT1"])   # deg

nx = int(np.ceil((ra_max - ra_min) / pixscale))
ny = int(np.ceil((dec_max - dec_min) / pixscale))


In [11]:

# =========================
# 4. 构建 FK5 赤道 WCS
#    关键点：RA 的 CDELT1 必须为负
# =========================
wcs_fk5 = WCS(naxis=2)
wcs_fk5.wcs.ctype = ["RA---CAR", "DEC--CAR"]
wcs_fk5.wcs.cunit = ["deg", "deg"]

wcs_fk5.wcs.cdelt = np.array([
    -pixscale,     # RA 向左增大
     pixscale
])

wcs_fk5.wcs.crpix = [1.0, 1.0]
wcs_fk5.wcs.crval = [
    ra_max,        # 注意：对应左边界
    dec_min
]

wcs_fk5.wcs.radesys = "FK5"
wcs_fk5.wcs.equinox = 2000.0


In [12]:

# =========================
# 5. Reproject
# =========================
array_eq, footprint = reproject_interp(
    (data, wcs_gal),
    wcs_fk5,
    shape_out=(ny, nx)
)


In [13]:

# =========================
# 6. 写 FITS（清理 header）
# =========================
out_hdr = wcs_fk5.to_header()

if out_hdr['CRVAL2'] != 0:
    out_hdr['CRPIX2'] -= out_hdr['CRVAL2'] / out_hdr['CDELT2']
    out_hdr['CRVAL2'] = 0

# 拷贝物理量
for key in ["BUNIT", "SPECSYS", "RESTFREQ", "BMAJ", "BMIN", "BPA", "TELESCOP"]:
    if key in hdr:
        out_hdr[key] = hdr[key]

out_hdr["BUNIT"] = "K"  # 更新单位

# 移除 BLANK / DATE-OBS（避免 warning）
out_hdr.pop("BLANK", None)
out_hdr.pop("DATE-OBS", None)


In [14]:

outfile = "HI4PI_RA60_80_DEC-13_2_HVC_moment_0.fits"
fits.PrimaryHDU(
    array_eq.astype(np.float32),
    header=out_hdr
).writeto(outfile, overwrite=True)

hdul.close()
print(f"Saved: {outfile}")


Saved: HI4PI_RA60_80_DEC-13_2_HVC_moment_0.fits
